<a href="https://colab.research.google.com/github/cwilson-su/Large-Scale-Data-Analytics-Labs/blob/main/LAB2_2026_Fenetres_avec_duckdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

révision : 02/10/2026

# TP2 — Fonctions de fenêtrage (`OVER`) avec DuckDB — ÉNONCÉ

* **Données** : un petit exemple (visites de photos), puis le jeu de données Foursquare (New York)
* **Moteur SQL** : DuckDB

Ce TP met en pratique la partie *Fenêtrage* du cours 2 (*Analyse multidimensionnelle en SQL*) :
`row_number`, `rank`, `dense_rank`, `PARTITION BY`, fenêtres croissantes et glissantes (`ROWS`, `RANGE`).

# Préparation

Lire la [documentation DuckDB SQL](https://duckdb.org/docs/sql/introduction.html) :
* [Functions](https://duckdb.org/docs/sql/functions/overview)
* [Window functions](https://duckdb.org/docs/sql/window_functions)

Affichage convivial des résultats d'une requête

In [ ]:
try:
    from itables import init_notebook_mode
except ImportError:
    !pip -q install itables
    from itables import init_notebook_mode

init_notebook_mode(all_interactive=True)

In [ ]:
import pandas as pd

def display(query, n=30):
    """Affiche les n premiers tuples d'une requête DuckDB sous forme de tableau."""
    return query.limit(n).df()

# Variante pour Colab uniquement :
# from google.colab import data_table
# def display(query, n=100):
#     return data_table.DataTable(query.limit(n).df(), include_index=False, num_rows_per_page=10)

## Installer et importer DuckDB

In [ ]:
try:
    import duckdb
except ImportError:
    !pip install duckdb
    import duckdb

Définir la commande magique **`%%sql`** pour écrire plus simplement une requête SQL dans une cellule.

In [ ]:
from IPython.core.magic import register_line_cell_magic

def removeComments(query):
    """Supprime les lignes de commentaire SQL (commençant par --)."""
    result = ""
    for line in query.split('\n'):
        if not line.strip().startswith("--"):
            result += line + "\n"
    return result

@register_line_cell_magic
def sql(line, cell=None):
    "Exécute une ou plusieurs requêtes SQL séparées par ';'. Usage : %%sql"
    val = cell if cell is not None else line
    derniere = None
    est_requete = False
    for r in removeComments(val).split(";"):
        r = r.strip()
        if len(r) > 2:
            derniere = db.sql(r)
            est_requete = r.lower().startswith(('select', 'with', 'from'))
    if est_requete:
        return display(derniere)
    print('ok')

print("fonction définie")

## Démarrer DuckDB

In [ ]:
db = duckdb.connect(':memory:')
# Les dates Foursquare sont en UTC : on fixe le fuseau horaire de la session
db.sql("SET TimeZone = 'UTC'")

# vérifier que le service fonctionne
display(db.sql("SELECT 'hello' AS col"))

In [ ]:
def display_schema(table):
    return display(db.sql(f"""
        SELECT column_name, data_type
        FROM duckdb_columns
        WHERE table_name = '{table}'
    """))

def display_all_views():
    return display(db.sql("SELECT view_name, sql, temporary FROM duckdb_views"))

print("fonctions définies")

# Partie 1 — Premier exemple

Un premier exemple pour comprendre les fonctions de fenêtrage. Ce sont les données utilisées dans le cours.

## Les données

### Les visites

In [ ]:
visite_attrs = ["photoId", "personId", "date", "lon", "lat", "note"]
visite_tuples = [("p1",  "Bob",    "03/09/2020", 41.5, 12.8, 3),
                 ("p2",  "Alice",  "01/09/2020", 41.6, 12.8, 5),
                 ("p3",  "Bob",    "04/09/2020", 41.6, 13.2, 1),
                 ("p4",  "Bob",    "04/09/2020", 40.1, 12.0, 2),
                 ("p5",  "Alice",  "04/09/2020", 40.1, 12.0, 2),
                 ("p6",  "Alice",  "05/09/2020", 41.6, 12.7, 4),
                 ("p7",  "Alice",  "05/10/2020", 41.8, 12.8, 4),
                 ("p8",  "Carole", "25/12/2019", 30.1, 10.1, 2),
                 ("p9",  "David",  "25/12/2019", 30.1, 10.1, 1),
                 ("p10", "Eva",    "25/12/2019", 30.1, 10.1, 5),
                 ("p11", "Eva",    "26/12/2019", 31.1, 10.1, 3),
                 ("p12", "Alice",  "01/02/2020", 32.1, 12.1, 3),
                 ("p13", "Bob",    "01/02/2020", 32.1, 12.1, 5),
                 ("p14", "Carole", "01/02/2020", 32.1, 12.1, 2),
                 ("p15", "Carole", "11/11/2019", 49.1, 10.1, 1),
                 ("p16", "Alice",  "25/12/2019", 30.1, 10.1, 4),
                 ("p17", "Alice",  "02/02/2020", 49.1, 10.1, 5),
                ]

visites_df = pd.DataFrame(visite_tuples, columns=visite_attrs)

db.sql("""
DROP TABLE IF EXISTS Visites CASCADE;

CREATE TABLE Visites AS
SELECT photoId,
       personId,
       strptime(date, '%d/%m/%Y') AS date_num,
       extract(year FROM date_num) AS année,
       extract(month FROM date_num) AS mois,
       lon,
       lat,
       note
FROM visites_df""")

Visites = db.table("Visites")
display(Visites)

In [ ]:
display_schema("Visites")

Exemple de cellule utilisant la commande `%%sql` :

In [ ]:
%%sql
SELECT *
FROM Visites
LIMIT 3

### Les lieux

In [ ]:
lieux_attributs = ["lon", "lat", "ville", "pays"]

lieux_tuples = [(41.5, 12.8, "Aix", "France"),
                (41.6, 12.8, "Aix", "France"),
                (41.6, 12.7, "Nice", "France"),
                (41.8, 12.8, "Marseille", "France"),
                (41.6, 13.2, "Rome", "Italie"),
                (40.1, 12.0, "Oslo", "Norvege"),
                (30.1, 10.1, "Paris", "France"),
                (31.1, 10.1, "StDenis", "France"),
                (32.1, 12.1, "Lille", "France"),
                (49.1, 10.1, "Pau", "France")]

lieux_df = pd.DataFrame(lieux_tuples, columns=lieux_attributs)
db.sql("""
DROP TABLE IF EXISTS Lieux;
CREATE TABLE Lieux AS
SELECT ville, pays, lon, lat FROM lieux_df""")

Lieux = db.table("Lieux")
display(Lieux)

### Les professions

In [ ]:
profession_attributs = ["personID", "profession"]

profession_tuples = [("Alice", "Architecte"),
                     ("Bob", "Data science"),
                     ("Carole", "Data science"),
                     ("David", "Architecte"),
                     ("Eva", "Vendeuse"),
                     ("Franck", "Vendeur"),
                     ("Greta", "Economiste")]

profession_df = pd.DataFrame(profession_tuples, columns=profession_attributs)
db.sql("""
DROP TABLE IF EXISTS Professions;
CREATE TABLE Professions AS
SELECT * FROM profession_df""")

Professions = db.table("Professions")
display(Professions)

### Le détail des visites

Définir une vue contenant le détail des visites.

In [ ]:
db.sql("""
CREATE OR REPLACE VIEW VisitesDetail AS
SELECT v.photoId, p.personID, p.profession, v.date_num, v.mois, v.année,
       v.lon, v.lat, l.ville, l.pays, v.note
FROM Visites v
JOIN Lieux l ON (v.lon = l.lon AND v.lat = l.lat)
JOIN Professions p ON (v.personId = p.personId)
""")

# Même requête écrite avec un NATURAL JOIN :
# SELECT ... FROM Visites v NATURAL JOIN Lieux l NATURAL JOIN Professions p

VisitesDetail = db.view('VisitesDetail')
display(VisitesDetail)

## Exemples de requêtes avec fenêtre

Ces requêtes sont celles du cours : exécutez-les et observez le résultat.

### Numéroter les tuples : `row_number()`

In [ ]:
%%sql
SELECT v.personId, v.photoId, v.note,
       row_number() OVER (ORDER BY personId, photoId) AS numeroPhoto
FROM Visites v

Remarque : `photoId` est une chaîne de caractères, l'ordre est donc lexicographique (`p12 < p16 < p2`).

### Classer les visites par note décroissante : `rank()`

In [ ]:
%%sql
SELECT v.personId, v.photoId, v.note,
       rank() OVER (ORDER BY note DESC) AS rang
FROM Visites v

### Requête top-K
Le top 5 des visites ayant les meilleures notes.
Il peut y avoir plus de 5 visites dans le résultat à cause des **ex æquo**.

In [ ]:
%%sql
WITH Classement AS (
  SELECT v.personId, v.photoId, v.note,
         rank() OVER (ORDER BY note DESC) AS rang
  FROM Visites v
)
SELECT *
FROM Classement
WHERE rang <= 5
ORDER BY rang

### Classement dense : `dense_rank()`

In [ ]:
%%sql
SELECT v.personId, v.photoId, v.note,
       dense_rank() OVER (ORDER BY note DESC) AS rang_dense
FROM Visites v

### Fenêtrage et partitionnement : `PARTITION BY`

Une partition regroupe les tuples ayant la même valeur pour les attributs de partitionnement.
On obtient une fenêtre par partition.

Le classement **par ville** des visites, de la mieux notée à la moins bien notée

In [ ]:
%%sql
SELECT v.photoId, v.personID, v.ville, v.note,
       rank() OVER (PARTITION BY ville ORDER BY note DESC) AS rang
FROM VisitesDetail v
ORDER BY ville, rang

Le classement **par pays** des villes ayant le plus de visites

In [ ]:
%%sql
SELECT pays, ville, count(*) AS nbVisite,
       rank() OVER (PARTITION BY pays ORDER BY count(*) DESC) AS rang
FROM VisitesDetail v
GROUP BY pays, ville
ORDER BY pays, rang

La fenêtre est évaluée **après** le `GROUP BY` : elle peut donc utiliser `count(*)`.

## Questions

### Question 1 : temps écoulé depuis la première visite

Pour chaque visite de chaque personne, calculer le temps écoulé depuis la **première** visite de cette personne.
Le temps écoulé est un intervalle de temps, obtenu par différence entre deux dates ; il s'exprime en jours.

Schéma attendu : `(photoId, personId, date_num, datePremiereVisite, jours_ecoules)`, trié par personne puis par date.

*Astuce* : commencer par compléter chaque visite avec la date de la première visite de la personne.

In [ ]:
%%sql
-- Écrire la requête ici

### Question 2 : fenêtre glissante de taille fixe — visite précédente

Pour chaque visite de chaque personne, calculer le nombre de jours écoulés depuis la visite **précédente** de cette personne.
Si une personne a effectué plusieurs visites le même jour, on considère ces visites dans l'ordre de leur `photoId`.
Pour la première visite d'une personne, `precedenteDate` et `nbJours` valent `NULL`.

Schéma attendu : `(photoId, personId, date_num, precedenteDate, nbJours)`

*Astuce* : la fonction `datediff('day', d1, d2)` donne le nombre de jours entre deux dates.

In [ ]:
%%sql
-- Écrire la requête ici

### Question 3 : semaine glissante

Pour chaque jour où une personne a effectué au moins une visite, calculer le nombre de visites de cette personne
sur les 7 derniers jours (le jour courant et les 6 jours précédents).

Schéma attendu : `(personId, date_num, nbVisites_7jours)`

*Astuce* : une fenêtre `RANGE` est définie par un écart de valeur, par exemple `INTERVAL 6 DAYS`.

In [ ]:
%%sql
-- Écrire la requête ici

### Question 4 : trajets

On veut enrichir les visites avec une information de trajet.
Un trajet regroupe des visites consécutives d'une même personne telles que deux visites consécutives
sont séparées d'**au plus 7 jours**. Une visite séparée de plus de 7 jours de la précédente commence un nouveau trajet.
Les trajets d'une personne sont numérotés 1, 2, 3, … par ordre chronologique.

Schéma attendu : `(photoId, personId, date_num, nbJours, trajet)`

*Astuces* :
* réutiliser la réponse à la question 2 ;
* une expression `if(condition, 1, 0)` (ou `CASE WHEN … THEN 1 ELSE 0 END`) indique si la visite commence un nouveau trajet ;
* une somme cumulée de ces indicateurs donne le numéro de trajet.

In [ ]:
%%sql
-- Écrire la requête ici

### Question 4b (bonus) : trajets d'au plus 3 jours

On ajoute une condition : un trajet ne peut pas durer plus de **3 jours** au total.
Une visite commence donc un nouveau trajet si elle est séparée de plus de 7 jours de la visite précédente,
**ou** si elle a lieu plus de 3 jours après la première visite du trajet en cours.

Schéma attendu : `(photoId, personId, date_num, debutTrajet, trajet)`

*Question préalable* : pourquoi ne peut-on pas résoudre cette question avec une simple somme cumulée comme à la question 4 ?

In [ ]:
%%sql
-- Écrire la requête ici

# Partie 2 — Données Foursquare

Données de *check-in* issues du réseau social Foursquare (ville de New York, avril 2012 – février 2013).
Source : [Kaggle](https://www.kaggle.com/datasets/chetanism/foursquare-nyc-and-tokyo-checkin-dataset).

In [ ]:
import os
local_dir = os.environ["HOME"] + "/data"
print("local_dir is", local_dir)
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

URL d'accès aux jeux de données

In [ ]:
from urllib import request
import zipfile

# URL du dossier PUBLIC_DATASET contenant des fichiers de données pour les TP
PUBLIC_DATASET_URL = "https://gitlab.lip6.fr/naacke/tp/-/raw/main/dataset"

# en cas de problème de téléchargement, aller directement sur l'URL ci-dessous
print("Les datasets disponibles :", PUBLIC_DATASET_URL.replace("/raw/", "/tree/"))


def download_file(url, local_path):
    if not os.path.exists(local_path):
        print("downloading", url, "to", local_path)
        request.urlretrieve(url, local_path)
    else:
        print(local_path, "already exists")


def unzip_file(local_dir, file):
    zip_filepath = os.path.join(local_dir, file)
    try:
        with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
            do_extract = False
            for f in zip_ref.namelist():
                if not os.path.exists(os.path.join(local_dir, f)):
                    do_extract = True
                    break
                print(os.path.join(local_dir, f), "unzipped file already exists")
            if do_extract:
                print("unzip", zip_filepath)
                zip_ref.extractall(local_dir)
    except Exception as e:
        print(f"Une erreur est survenue : {e}")

In [ ]:
zip_filename = "dataset_TSMC2014_NYC.zip"
download_file(f"{PUBLIC_DATASET_URL}/foursquare/{zip_filename}", os.path.join(local_dir, zip_filename))
unzip_file(local_dir, zip_filename)

os.listdir(local_dir)

In [ ]:
# afficher les premières lignes du fichier
with open(local_dir + "/dataset_TSMC2014_NYC.csv", 'r') as data_file:
    for i in range(4):
        print(data_file.readline().strip())

## Table Visit

In [ ]:
db.sql(f"""
DROP TABLE IF EXISTS Visit;
CREATE TABLE Visit AS
  SELECT *
  FROM '{local_dir}/dataset_TSMC2014_NYC.csv';
""")
print("done")

In [ ]:
visit = db.table('Visit')
display(visit, 5)

In [ ]:
display_schema('Visit')

## Nettoyage

Certains utilisateurs ont plusieurs visites au **même instant** (du même POI ou de POI différents), ce qui n'est pas cohérent.

In [ ]:
%%sql
SELECT userId, utcTimestamp, count(*) AS nb_visites, count(DISTINCT venueId) AS nb_distinct_poi
FROM Visit
GROUP BY userId, utcTimestamp
HAVING nb_visites > 1
ORDER BY nb_visites DESC
LIMIT 10

Plusieurs visites du même lieu pour un utilisateur donné à un instant donné :

In [ ]:
%%sql
SELECT *
FROM Visit
WHERE userId = 344
  AND utcTimestamp = 'Mon Nov 19 16:38:14 +0000 2012'

Plusieurs visites de lieux différents pour un utilisateur donné à un instant donné :

In [ ]:
%%sql
SELECT *
FROM Visit
WHERE userId = 739
  AND utcTimestamp = 'Thu Apr 19 12:00:12 +0000 2012'

### Question 0 : dédoublonnage

Ne conserver qu'**une seule** visite par couple `(userId, utcTimestamp)` : celle dont le `venueId` est le plus petit.
Le résultat a le même schéma que la table `Visit`.

*Indication* : numéroter les visites à l'intérieur de chaque groupe `(userId, utcTimestamp)` avec `row_number()`.

In [ ]:
visit_no_duplicate = db.sql("""
-- Écrire la requête ici
""")
display(visit_no_duplicate, 2)

Vérifier qu'il n'y a plus de doublons (le résultat doit être vide) :

In [ ]:
display(db.sql("""
SELECT userId, utcTimestamp, count(*) AS nb
FROM visit_no_duplicate
GROUP BY userId, utcTimestamp
HAVING nb > 1
"""))

### Remplacer la table Visit par celle sans doublons

In [ ]:
nb_visites_avant = db.sql("SELECT count(*) FROM Visit").fetchone()[0]
print("nb de visites avant d'enlever les doublons :", nb_visites_avant)

db.sql("CREATE OR REPLACE TABLE Visit AS SELECT * FROM visit_no_duplicate")

nb_visites_apres = db.sql("SELECT count(*) FROM Visit").fetchone()[0]
print("nb de visites après avoir enlevé les doublons :", nb_visites_apres)

## Préparation des requêtes

Convertir le timestamp (chaîne de caractères) en date.

In [ ]:
visit2 = db.sql("""
  SELECT *, strptime(utcTimestamp, '%a %b %d %X %z %Y') AS datetime
  FROM Visit
""")
display(visit2)

## Exercice 1 : séquences de visites

### 1.0) Numéro de POI

Les identifiants `venueId` sont longs et peu lisibles. Définir :
* `venueId_poi(venueId, poi)` qui associe à chaque `venueId` un numéro `poi` allant de 1 à *n* ;
  *indication* : utiliser `row_number() OVER (…)` ;
* `visit_poi(userId, poi, datetime)`, les visites avec les numéros de POI.

In [ ]:
venueId_poi = db.sql("""
-- Écrire la requête ici
""")
display(db.sql("SELECT * FROM venueId_poi ORDER BY poi"))

In [ ]:
db.sql("SELECT count(*) AS nb_poi FROM venueId_poi")

In [ ]:
visit_poi = db.sql("""
-- Écrire la requête ici
""")
display(db.sql("SELECT * FROM visit_poi ORDER BY datetime"))

### 1.1) Rang

Pour chaque utilisateur, ordonner les visites par date et leur attribuer un **rang** allant de 1 (la plus ancienne) à *n* (la plus récente).

Schéma attendu : `visit_rank(userId, poi, datetime, rank)`

In [ ]:
visit_rank = db.sql("""
-- Écrire la requête ici
""")
display(visit_rank)

### 1.2) Date précédente et durée de déplacement

On veut connaître la durée séparant deux visites consécutives d'un même utilisateur.
Pour chaque visite, ajouter l'attribut **`prev_date`** contenant la date de la visite précédente du même utilisateur.
Pour la première visite d'un utilisateur, `prev_date` est égale à la date de la visite.

Puis ajouter l'attribut **`interval_duration`**, différence en secondes entre la date courante et `prev_date`.

*Indication* : voir la fonction `date_diff('second', a, b)`.

Schémas attendus : `visit_prev_date(userId, poi, datetime, rank, prev_date)`
puis `visit_duration(userId, poi, datetime, rank, prev_date, interval_duration)`

In [ ]:
visit_prev_date = db.sql("""
-- Écrire la requête ici
""")
display(visit_prev_date)

In [ ]:
visit_duration = db.sql("""
-- Écrire la requête ici
""")
display(visit_duration)

### 1.3) Numéro de trajectoire

Découper les visites de chaque utilisateur en trajectoires : dans une trajectoire, la durée entre deux visites
consécutives ne dépasse pas **20 heures** (20 × 3600 secondes).

Ajouter l'attribut **`debut`** valant 1 si la visite commence une trajectoire, 0 sinon,
puis l'attribut **`traj`** indiquant le numéro de la trajectoire (1, 2, …) pour un utilisateur donné.

*Indication* : utiliser une expression `CASE WHEN … THEN … ELSE … END`.

Schémas attendus : `visit_traj1(…, debut)` puis `visit_traj2(…, debut, traj)`

In [ ]:
visit_traj1 = db.sql("""
-- Écrire la requête ici
""")
display(visit_traj1)

In [ ]:
visit_traj2 = db.sql("""
-- Écrire la requête ici
""")
display(visit_traj2)

## Exercice 2 : fenêtres glissantes

### 2.1) Nombre de POI visités dans la semaine précédente

Ajouter l'attribut **`nb_POI_7jours`** donnant le nombre de POI **distincts** visités par l'utilisateur pendant
les 7 jours qui précèdent le jour de la visite courante. Les visites du jour courant ne sont **pas** comptées.

Exemple : pour une visite du 10 avril, on compte les POI visités du 3 au 9 avril inclus.

*Indication* : la fenêtre porte sur des **jours** (`datetime::date`) ; sa borne supérieure précède d'un jour la date courante.

Schéma attendu : `visit_poi_7(userId, poi, datetime, nb_POI_7jours)`

In [ ]:
visit_poi_7 = db.sql("""
-- Écrire la requête ici
""")
display(visit_poi_7)

### 2.2) Nombre cumulé de visites et de POI par utilisateur

Pour chaque visite, ajouter :
* **`cumul_visites`** : le nombre de visites effectuées par l'utilisateur **avant** la visite courante (0 pour la première) ;
* **`cumul_poi`** : le nombre de POI **distincts** déjà visités par l'utilisateur avant la visite courante.

Schéma attendu : `visit_cumul(userId, poi, datetime, rank, cumul_visites, cumul_poi)`

In [ ]:
visit_cumul = db.sql("""
-- Écrire la requête ici
""")
display(visit_cumul)

## Exercice 3 : visites dans une trajectoire

### 3.1) Fusionner les visites consécutives d'un même POI

Une trajectoire peut contenir plusieurs visites **consécutives** d'un même POI : on veut les fusionner en une seule visite.
Pour chaque trajectoire, regrouper les visites consécutives d'un même POI et calculer pour chaque visite fusionnée
sa date de début `date_debut`, sa date de fin `date_fin`, le nombre de check-ins `nb_checkins` et la durée `visit_duration` (en secondes).

* Rmq 1 : si un POI n'a pas de visites consécutives, ses dates de début et de fin sont identiques.
* Rmq 2 : une trajectoire peut contenir plusieurs visites **non consécutives** d'un même POI : elles ne doivent pas être fusionnées.

Schéma attendu : `trajectoire(userId, traj, visitId, poi, nb_checkins, date_debut, date_fin, visit_duration)`,
où `visitId` numérote les visites fusionnées dans la trajectoire.

*Démarche suggérée* : (1) récupérer le POI de la visite précédente dans la trajectoire ;
(2) marquer les visites qui commencent une nouvelle visite fusionnée ; (3) numéroter par somme cumulée ; (4) agréger.

In [ ]:
trajectoire = db.sql("""
-- Écrire la requête ici (on peut la décomposer en plusieurs étapes)
""")
display(trajectoire)

Afficher les visites fusionnées de l'utilisateur 984 :

In [ ]:
display(db.sql("""
SELECT *
FROM trajectoire
WHERE userId = 984
ORDER BY traj, visitId
"""))

### 3.2) Trajectoires contenant une fusion

Afficher toutes les visites (fusionnées) des trajectoires qui contiennent **au moins une** visite fusionnée
(c'est-à-dire avec `nb_checkins > 1`). Trier le résultat par `userId`, `traj`, `date_debut`.

*Indication* : une fenêtre `PARTITION BY userId, traj` permet de calculer une information sur toute la trajectoire
sans perdre le détail des visites.

In [ ]:
%%sql
-- Écrire la requête ici

### 3.3) Statistiques sur les trajectoires

a) Pour chaque trajectoire, calculer le nombre de visites (fusionnées) `nb_visite` et le nombre de POI distincts `nb_poi`.
Stocker le résultat dans une table `stat_traj(userId, traj, nb_visite, nb_poi)`.

b) Calculer le nombre moyen de visites et de POI par trajectoire.

c) Pour chaque valeur de `nb_visite` inférieure à 20, donner le nombre de trajectoires `nb_traj`
et le nombre moyen de POI `avg_nb_poi`, triés par `nb_visite` décroissant.

In [ ]:
# a)
db.sql("""
-- Écrire la requête ici
""")

In [ ]:
%%sql
-- b) Écrire la requête ici

In [ ]:
%%sql
-- c) Écrire la requête ici